# LangChain RAG - Config-Driven Version

This notebook loads all settings from config.yaml

In [12]:
# Load configuration from config.yaml
import yaml
from pathlib import Path

config_path = Path("../config/config.yaml")
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

print("📋 Configuration loaded:")
print(f"  State: {config['paths']['state']}")
print(f"  Year: {config['paths']['year']}")
print(f"  Document type: {config['paths']['document_type']}")
print(f"  Raw data: {config['paths']['raw_data']}")
print(f"  Chunk size: {config['chunking']['chunk_size']}")
print(f"  Embedding model: {config['embedding']['model_name']}")
print(f"  LLM: {config['llm']['model']}")

📋 Configuration loaded:
  State: NC
  Year: 2026
  Document type: formulary
  Raw data: data/raw/NC/2026/formulary
  Chunk size: 500
  Embedding model: sentence-transformers/all-MiniLM-L6-v2
  LLM: llama3.1


In [3]:
# Load PDF documents
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader

loader = DirectoryLoader(
    f"../{config['paths']['raw_data']}",
    glob="*.pdf",
    loader_cls=PyPDFLoader
)

print(f"Loading PDFs from: {config['paths']['raw_data']}")
all_docs = loader.load()
print(f"✅ Loaded {len(all_docs)} pages")

/tmp/ipykernel_11699/3427719367.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
/home/bhargav/portfolio-project/healthins-enrollment-assistant-dev/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading PDFs from: data/raw/NC/2026/formulary
✅ Loaded 133 pages


In [4]:
# Chunk documents
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=config['chunking']['chunk_size'],
    chunk_overlap=config['chunking']['chunk_overlap'],
    length_function=len,
    is_separator_regex=False,
)

chunks = text_splitter.split_documents(all_docs)
print(f"✅ Created {len(chunks)} chunks")
print(f"  Chunk size: {config['chunking']['chunk_size']}")
print(f"  Chunk overlap: {config['chunking']['chunk_overlap']}")

✅ Created 774 chunks
  Chunk size: 500
  Chunk overlap: 50


In [5]:
# Create embeddings and vector store
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name=config['embedding']['model_name']
)

print(f"Creating vector store with {config['embedding']['model_name']}...")
vectorstore = FAISS.from_documents(chunks, embeddings)
print(f"✅ Vector store created with {len(chunks)} chunks")

/tmp/ipykernel_11699/2038989982.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1005.55it/s]


Creating vector store with sentence-transformers/all-MiniLM-L6-v2...
✅ Vector store created with 774 chunks


In [10]:
# Check Ollama is running locally (no API key needed)
import requests

try:
    requests.get("http://localhost:11434", timeout=2)
    print("✅ Ollama is running")
except requests.exceptions.ConnectionError:
    print("❌ Ollama not reachable - run 'ollama serve'")

✅ Ollama is running


In [13]:
# Initialize LLM
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOllama(
    model=config['llm']['model'],
    temperature=config['llm']['temperature'],
    num_predict=config['llm']['max_tokens']
)

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant answering questions about health insurance plans. Use only the provided context to answer."),
    ("user", "Context:\n{context}\n\nQuestion: {question}\n\nAnswer:")
])

print(f"✅ LLM configured:")
print(f"  Model: {config['llm']['model']}")
print(f"  Temperature: {config['llm']['temperature']}")

✅ LLM configured:
  Model: llama3.1
  Temperature: 0.3


In [14]:
# RAG function
def ask_question(question, k=None):
    if k is None:
        k = config['retrieval']['top_k']
    
    docs = vectorstore.similarity_search(question, k=k)
    context = "\n\n".join([doc.page_content for doc in docs])
    
    messages = prompt_template.format_messages(
        context=context,
        question=question
    )
    
    response = llm.invoke(messages)
    return response.content

print(f"✅ RAG function ready (top_k={config['retrieval']['top_k']})")

✅ RAG function ready (top_k=3)


In [15]:
# Test question
answer = ask_question("What pharmacy benefits are covered?")
print(answer)

According to the provided text, the pharmacy benefits are covered under the Pharmacy Benefit portion of the Oscar plan, not the Medical Benefit portion. However, it does not provide a detailed list of covered pharmacy benefits.

It does mention that if a specific drug is not included in the formulary, you should first contact Concierge and ask if your drug is covered.


In [20]:
# Naming exact drug strengths (not just the drug class) keys retrieval into the precise table rows
specific_question = (
    "What is the tier and quantity limit for acetaminophen with codeine "
    "solution 120-12 mg/5ml and tablet 300-15 mg?"
)
answer = ask_question(specific_question, k=5)
print(answer)

According to the provided context, the tier and quantity limit for acetaminophen with codeine solution 120-12 mg/5ml is:

* Tier 1 QL (2700 ml every 30 days)
* Subject to initial 5-day limit for 19 and younger; 7-day initial limit for all other ages

And for the acetaminophen with codeine tablet 300-15 mg:

* Tier 1 QL (390 tabs every 30 days)
* Subject to initial 5-day limit for 19 and younger; 7-day initial limit for all other ages
